# 🧬 Enhancer Repair Research: AI-Driven Regulatory Grammar Discovery

**Decoding Regulatory Grammar Through Neural Network Repair**

**A100 GPU Optimized Version** (~280M Parameters)

This notebook implements a novel methodology for discovering gene regulatory grammar by using neural network "repair" as a biological probe.

## Research Questions
1. **Cross-Species Conservation:** Which regulatory motifs are conserved across vertebrate evolution?
2. **Combinatorial Grammar:** Which motifs work synergistically versus independently?
3. **Generative Understanding:** Can the model create functional enhancers from scratch?

## Hardware Requirements
- **Recommended:** Google Colab Pro with A100 GPU (80GB VRAM)
- **Compatible:** V100 (32GB), T4 (16GB) with automatic config scaling

---
**Important:** Run cells in order. Use `Runtime > Run all` for full execution.

## 1. Environment Setup

### 1.1 Check GPU Availability and Auto-Configure

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Check GPU
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device('cuda')
else:
    print("WARNING: No GPU detected. Training will be slow.")
    device = torch.device('cpu')

print(f"\nUsing device: {device}")

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")

# Check GPU
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    device = torch.device('cuda')
    
    # Auto-detect GPU type for optimal config
    if 'A100' in gpu_name:
        GPU_TYPE = 'A100'
        print(f"\n🚀 Detected A100 GPU - Using maximum model size (~280M params)")
    elif 'V100' in gpu_name:
        GPU_TYPE = 'V100'
        print(f"\n⚡ Detected V100 GPU - Using large model size (~120M params)")
    elif 'T4' in gpu_name:
        GPU_TYPE = 'T4'
        print(f"\n💻 Detected T4 GPU - Using medium model size (~27M params)")
    else:
        GPU_TYPE = 'OTHER'
        print(f"\n🔧 Unknown GPU - Using medium model size (~27M params)")
else:
    print("\n⚠️  WARNING: No GPU detected. Training will be slow.")
    device = torch.device('cpu')
    GPU_TYPE = 'CPU'

print(f"\nUsing device: {device}")

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Create project directory
    PROJECT_DIR = '/content/drive/MyDrive/enhancer_repair_research'
else:
    PROJECT_DIR = '.'

import os
from pathlib import Path

# Create directory structure
dirs = ['data/raw', 'data/processed', 'models', 'checkpoints', 
        'results/cross_species', 'results/grammar', 'results/generative', 'results/tables',
        'figures', 'logs']

for d in dirs:
    Path(f"{PROJECT_DIR}/{d}").mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print("Directory structure created.")

### 1.3 Install Dependencies

In [ ]:
if IN_COLAB:
    print("Installing additional dependencies...")
    !pip install -q biopython==1.81 captum==0.6.0
    print("Dependencies installed.")

# Verify imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
import json
import time
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

## 2. GPU-Specific Configuration

Automatically selects optimal model size based on detected GPU.

In [ ]:
# GPU-Specific Model Configurations
GPU_CONFIGS = {
    'A100': {
        # Model Architecture (~280M parameters)
        'n_transformer_layers': 24,
        'n_attention_heads': 16,
        'd_model': 1024,
        'dim_feedforward': 4096,
        'dropout': 0.1,
        'cnn_channels': [512, 768, 1024, 1024],
        'cnn_kernels': [7, 11, 15, 19],
        
        # Training
        'batch_size': 256,
        'num_epochs': 50,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 1,
        'samples_per_epoch': 500000,
        'use_mixed_precision': True,
        
        # Optimization
        'max_grad_norm': 0.5,
        'scheduler_factor': 0.5,
        'scheduler_patience': 3,
        'scheduler_min_lr': 1e-6,
        'early_stopping_patience': 10,
        'label_smoothing': 0.1,
    },
    
    'V100': {
        # Model Architecture (~120M parameters)
        'n_transformer_layers': 16,
        'n_attention_heads': 12,
        'd_model': 768,
        'dim_feedforward': 3072,
        'dropout': 0.1,
        'cnn_channels': [384, 512, 768],
        'cnn_kernels': [7, 11, 15],
        
        # Training
        'batch_size': 128,
        'num_epochs': 40,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 2,
        'samples_per_epoch': 300000,
        'use_mixed_precision': True,
        
        # Optimization
        'max_grad_norm': 0.5,
        'scheduler_factor': 0.5,
        'scheduler_patience': 3,
        'scheduler_min_lr': 1e-6,
        'early_stopping_patience': 8,
        'label_smoothing': 0.1,
    },
    
    'T4': {
        # Model Architecture (~27M parameters)
        'n_transformer_layers': 10,
        'n_attention_heads': 8,
        'd_model': 256,
        'dim_feedforward': 1536,
        'dropout': 0.1,
        'cnn_channels': [128, 256, 384],
        'cnn_kernels': [7, 11, 15],
        
        # Training
        'batch_size': 64,
        'num_epochs': 30,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 4,
        'samples_per_epoch': 100000,
        'use_mixed_precision': True,
        
        # Optimization
        'max_grad_norm': 0.5,
        'scheduler_factor': 0.5,
        'scheduler_patience': 3,
        'scheduler_min_lr': 1e-6,
        'early_stopping_patience': 7,
        'label_smoothing': 0.0,
    },
    
    'OTHER': {  # Default fallback
        'n_transformer_layers': 10,
        'n_attention_heads': 8,
        'd_model': 256,
        'dim_feedforward': 1536,
        'dropout': 0.1,
        'cnn_channels': [128, 256, 384],
        'cnn_kernels': [7, 11, 15],
        'batch_size': 64,
        'num_epochs': 30,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'gradient_accumulation_steps': 4,
        'samples_per_epoch': 100000,
        'use_mixed_precision': False,
        'max_grad_norm': 0.5,
        'scheduler_factor': 0.5,
        'scheduler_patience': 3,
        'scheduler_min_lr': 1e-6,
        'early_stopping_patience': 7,
        'label_smoothing': 0.0,
    }
}

# Select configuration based on detected GPU
CONFIG = GPU_CONFIGS.get(GPU_TYPE, GPU_CONFIGS['OTHER'])

# Add common settings
CONFIG.update({
    'sequence_length': 200,
    'num_classes': 2,
    'num_workers': 2,
    'pin_memory': True,
    'project_dir': PROJECT_DIR,
    'checkpoint_dir': f"{PROJECT_DIR}/checkpoints",
    'data_dir': f"{PROJECT_DIR}/data/processed",
    'figures_dir': f"{PROJECT_DIR}/figures",
    'results_dir': f"{PROJECT_DIR}/results",
})

# Print selected configuration
print(f"\n{'='*70}")
print(f"SELECTED CONFIGURATION: {GPU_TYPE}")
print(f"{'='*70}")
print(f"Model Parameters:")
print(f"  Transformer Layers: {CONFIG['n_transformer_layers']}")
print(f"  Attention Heads: {CONFIG['n_attention_heads']}")
print(f"  Model Dimension: {CONFIG['d_model']}")
print(f"  Feedforward Dim: {CONFIG['dim_feedforward']}")
print(f"  CNN Channels: {CONFIG['cnn_channels']}")
print(f"\nTraining Settings:")
print(f"  Batch Size: {CONFIG['batch_size']}")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Samples/Epoch: {CONFIG['samples_per_epoch']:,}")
print(f"  Mixed Precision: {CONFIG['use_mixed_precision']}")
print(f"  Gradient Accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"{'='*70}")

# Research-specific configurations
CROSS_SPECIES_CONFIG = {
    'n_sequences_per_species': 300,
    'damage_fraction': 0.1,
    'max_repair_iterations': 50,
    'repair_learning_rate': 0.01,
    'target_probability': 0.9,
}

GRAMMAR_CONFIG = {
    'n_pairs_to_test': 500,
    'n_sequences_per_pair': 3,
    'n_sequences_for_motifs': 100,
    'importance_percentile': 75,
}

GENERATIVE_CONFIG = {
    'n_generation_attempts': 200,
    'max_iterations': 500,
    'target_probability': 0.80,
    'learning_rate': 0.05,
}

print("\n✓ All configurations loaded")

## 3. Data Acquisition

### 3.1 DNA Encoding Functions

**Note:** This notebook can use either real ENCODE/Ensembl data or synthetic data.
Set `USE_REAL_DATA = True` for publication-quality research (requires internet).

In [ ]:
# Choose data source
USE_REAL_DATA = True  # Set to False to use synthetic data

def one_hot_encode(sequence):
    """Convert DNA string to one-hot encoded array (4, seq_len)."""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq_len = len(sequence)
    encoded = np.zeros((4, seq_len), dtype=np.float32)
    for i, nucleotide in enumerate(sequence.upper()):
        if nucleotide in mapping:
            encoded[mapping[nucleotide], i] = 1.0
    return encoded

def decode_sequence(one_hot):
    """Convert one-hot array back to DNA string."""
    mapping = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}
    seq_len = one_hot.shape[1]
    return ''.join([mapping[np.argmax(one_hot[:, i])] for i in range(seq_len)])

def calculate_gc_content(sequence):
    """Calculate GC content of a DNA sequence."""
    sequence = sequence.upper()
    gc_count = sequence.count('G') + sequence.count('C')
    return gc_count / len(sequence) if len(sequence) > 0 else 0.0

# Test
test_seq = "ATGCGTACGATCGATCGTAGCTAGCTACGATCGATCG"
encoded = one_hot_encode(test_seq)
decoded = decode_sequence(encoded)
print(f"Original: {test_seq}")
print(f"Decoded:  {decoded}")
print(f"Match: {test_seq == decoded}")
print(f"GC Content: {calculate_gc_content(test_seq):.2%}")
print(f"\n✓ Encoding functions working correctly")

if USE_REAL_DATA:
    print("\n⚠️  Real data mode enabled - will download from ENCODE/Ensembl")
else:
    print("\n💡 Synthetic data mode - using generated sequences for demo")

### 3.2 Generate or Download Data

Optimized for each GPU type. Larger datasets for A100.

In [ ]:
import time
from pathlib import Path
import random
import gzip
import tempfile
import urllib.request

def download_encode_coordinates(species='human', max_sequences=50000):
    """Download ENCODE cCRE coordinates with fallback URLs."""
    # Multiple URL sources with fallbacks - SCREEN API V13 is the current stable source
    url_sources = {
        'human': [
            # SCREEN API V13 (current stable)
            'https://api.wenglab.org/screen_v13/fdownloads/cCREs/GRCh38-cCREs.bed.gz',
            # ENCODE Portal direct download
            'https://www.encodeproject.org/files/ENCFF334DFY/@@download/ENCFF334DFY.bed.gz',
            # Legacy V3 registry (may be deprecated)
            'https://downloads.wenglab.org/Registry/V3/GRCh38-cCREs.bed.gz',
        ],
        'mouse': [
            # SCREEN API V13 (current stable)
            'https://api.wenglab.org/screen_v13/fdownloads/cCREs/mm10-cCREs.bed.gz',
            # ENCODE Portal direct download
            'https://www.encodeproject.org/files/ENCFF056LNT/@@download/ENCFF056LNT.bed.gz',
            # Legacy V3 registry (may be deprecated)
            'https://downloads.wenglab.org/Registry/V3/mm10-cCREs.bed.gz',
        ]
    }
    
    if species not in url_sources:
        raise ValueError(f"Species must be 'human' or 'mouse', got {species}")
    
    print(f"Downloading {species} ENCODE data...")
    
    # Try each URL in order until one succeeds
    temp_file_path = None
    download_success = False
    errors = []  # Track all errors for better debugging
    
    for url in url_sources[species]:
        print(f"Trying: {url}")
        # Clean up any previous temp file from failed attempts
        if temp_file_path and Path(temp_file_path).exists():
            try:
                Path(temp_file_path).unlink()
            except OSError:
                pass
            temp_file_path = None
        
        try:
            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.bed.gz')
            temp_file_path = temp_file.name
            temp_file.close()  # Close so urlretrieve can write to it
            urllib.request.urlretrieve(url, temp_file_path)
            # Verify the file is valid BED format gzipped data
            with gzip.open(temp_file_path, 'rt') as f:
                for line in f:
                    if line.startswith('#') or line.startswith('track'):
                        continue
                    parts = line.strip().split('\t')
                    # Valid BED format should have at least 3 columns
                    if len(parts) >= 3:
                        int(parts[1])  # Verify start is an integer
                        int(parts[2])  # Verify end is an integer
                        download_success = True
                        print(f"✓ Successfully downloaded from {url}")
                        break
                if download_success:
                    break
        except Exception as e:
            errors.append(f"{url}: {e}")
            print(f"⚠️  Failed: {e}")
            continue
    
    if not download_success:
        # Clean up temp file on failure
        if temp_file_path and Path(temp_file_path).exists():
            try:
                Path(temp_file_path).unlink()
            except OSError:
                pass
        print(f"⚠️  All download sources failed")
        for err in errors:
            print(f"  - {err}")
        return []
    
    print("Parsing BED file...")
    coordinates = []
    
    try:
        with gzip.open(temp_file_path, 'rt') as f:
            for line in tqdm(f, desc="Reading coordinates"):
                if line.startswith('#') or line.startswith('track'):
                    continue
                parts = line.strip().split('\t')
                if len(parts) >= 3:
                    chrom, start, end = parts[0], int(parts[1]), int(parts[2])
                    length = end - start
                    if 150 <= length <= 250:
                        coordinates.append({
                            'chrom': chrom, 'start': start, 'end': end, 'length': length
                        })
                if len(coordinates) >= max_sequences * 2:
                    break
    finally:
        # Clean up temp file
        if temp_file_path and Path(temp_file_path).exists():
            try:
                Path(temp_file_path).unlink()
            except OSError:
                pass
    
    # Random sample
    if len(coordinates) > max_sequences:
        random.seed(42)
        coordinates = random.sample(coordinates, max_sequences)
    
    print(f"Downloaded {len(coordinates):,} {species} coordinates")
    return coordinates

def fetch_sequence_ensembl(chrom, start, end, species='human', max_retries=3):
    """Fetch DNA sequence from Ensembl REST API."""
    species_map = {
        'human': 'homo_sapiens',
        'mouse': 'mus_musculus',
        'zebrafish': 'danio_rerio',
        'chicken': 'gallus_gallus'
    }
    
    species_name = species_map.get(species, 'homo_sapiens')
    chrom_clean = chrom.replace('chr', '')
    
    url = (f"https://rest.ensembl.org/sequence/region/"
           f"{species_name}/{chrom_clean}:{start}-{end}?content-type=text/plain")
    
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                sequence = response.read().decode('utf-8').strip().upper()
                return sequence
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
            return None
    return None


def download_sequences_api(coordinates, species, target_length=200):
    """Download sequences using Ensembl API with rate limiting."""
    sequences = []
    failed = 0
    
    print(f"Downloading {len(coordinates):,} sequences via Ensembl API")
    print(f"Rate limit: ~15 requests/second")
    print(f"Estimated time: {len(coordinates) / 15 / 60:.1f} minutes")
    
    for i, coord in enumerate(tqdm(coordinates, desc=f"{species} sequences")):
        seq = fetch_sequence_ensembl(
            coord['chrom'], coord['start'], coord['end'], species
        )
        
        if seq and len(seq) >= 150 and 'N' not in seq:
            # Standardize length
            if len(seq) > target_length:
                seq = seq[:target_length]
            elif len(seq) < target_length:
                seq = seq + 'A' * (target_length - len(seq))
            sequences.append(seq)
        else:
            failed += 1
        
        # Rate limiting
        if (i + 1) % 15 == 0:
            time.sleep(1)
    
    print(f"Downloaded {len(sequences):,} sequences (failed: {failed:,})")
    return sequences


def download_encode_bed(species='human', max_sequences=50000):
    """Download ENCODE cCRE coordinates and sequences (wrapper for backward compatibility)."""
    if not USE_REAL_DATA:
        return []
    
    coords = download_encode_coordinates(species, max_sequences)
    if len(coords) == 0:
        return []
    
    # Download a smaller subset for sequences to avoid timeout
    sample_size = min(max_sequences, len(coords))
    coords_sample = random.sample(coords, sample_size) if len(coords) > sample_size else coords
    
    seqs = download_sequences_api(coords_sample, species)
    return seqs

print("✓ Data download functions defined")


def generate_synthetic_enhancer(length=200):
    """Generate synthetic enhancer-like sequence with motifs."""
    # Known motifs that make sequences more enhancer-like
    motifs = ['TATAAA', 'CACGTG', 'GGGCGG', 'CCAAT', 'GCGCGC', 'ATGCAT']
    
    # Start with random sequence (higher GC content like real enhancers)
    gc_prob = 0.45
    nucleotides = ['A', 'C', 'G', 'T']
    probs = [(1-gc_prob)/2, gc_prob/2, gc_prob/2, (1-gc_prob)/2]
    seq = ''.join(random.choices(nucleotides, weights=probs, k=length))
    seq = list(seq)
    
    # Insert 2-4 motifs at random positions
    n_motifs = random.randint(2, 4)
    for _ in range(n_motifs):
        motif = random.choice(motifs)
        pos = random.randint(10, length - len(motif) - 10)
        for i, nt in enumerate(motif):
            if pos + i < length:
                seq[pos + i] = nt
    
    return ''.join(seq)


def generate_background_sequence(length=200):
    """Generate random background sequence."""
    gc_prob = 0.42  # Genome average
    nucleotides = ['A', 'C', 'G', 'T']
    probs = [(1-gc_prob)/2, gc_prob/2, gc_prob/2, (1-gc_prob)/2]
    return ''.join(random.choices(nucleotides, weights=probs, k=length))


print("✓ Data download and synthetic generation functions defined")


In [ ]:
# Dataset sizes based on GPU type
if GPU_TYPE == 'A100':
    N_HUMAN = 30000
    N_MOUSE = 20000
    N_ZEBRAFISH = 15000
    N_CHICKEN = 15000
elif GPU_TYPE == 'V100':
    N_HUMAN = 20000
    N_MOUSE = 15000
    N_ZEBRAFISH = 10000
    N_CHICKEN = 10000
else:  # T4 and others
    N_HUMAN = 15000
    N_MOUSE = 10000
    N_ZEBRAFISH = 5000
    N_CHICKEN = 5000

N_BACKGROUND = N_HUMAN + N_MOUSE + N_ZEBRAFISH + N_CHICKEN

print(f"\n{'='*70}")
print(f"GENERATING DATASET FOR {GPU_TYPE}")
print(f"{'='*70}")
print(f"Target sizes:")
print(f"  Human: {N_HUMAN:,}")
print(f"  Mouse: {N_MOUSE:,}")
print(f"  Zebrafish: {N_ZEBRAFISH:,}")
print(f"  Chicken: {N_CHICKEN:,}")
print(f"  Background: {N_BACKGROUND:,}")
print(f"  TOTAL: {N_HUMAN + N_MOUSE + N_ZEBRAFISH + N_CHICKEN + N_BACKGROUND:,}")

np.random.seed(42)
random.seed(42)

if USE_REAL_DATA:
    print("\nDownloading real ENCODE data...")
    try:
        human_coords = download_encode_coordinates('human', N_HUMAN)
        mouse_coords = download_encode_coordinates('mouse', N_MOUSE)
        
        if len(human_coords) > 0 and len(mouse_coords) > 0:
            print("\nDownloading sequences via Ensembl API...")
            human_seqs = download_sequences_api(human_coords, 'human')
            mouse_seqs = download_sequences_api(mouse_coords, 'mouse')
            
            # For other species, use smaller synthetic datasets for now
            zebrafish_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_ZEBRAFISH), desc="Zebrafish (synthetic)")]
            chicken_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_CHICKEN), desc="Chicken (synthetic)")]
            background_seqs = [generate_background_sequence() for _ in tqdm(range(N_BACKGROUND), desc="Background")]
            
            if len(human_seqs) >= N_HUMAN // 2 and len(mouse_seqs) >= N_MOUSE // 2:
                print(f"\n✓ Downloaded real data: {len(human_seqs)} human, {len(mouse_seqs)} mouse sequences")
            else:
                print("\n⚠️  Insufficient real data downloaded, falling back to synthetic...")
                raise ValueError("Insufficient data")
        else:
            print("\n⚠️  ENCODE download failed, falling back to synthetic...")
            raise ValueError("Download failed")
    except Exception as e:
        print(f"Error: {e}")
        print("Generating synthetic dataset as fallback...")
        human_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_HUMAN), desc="Human")]
        mouse_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_MOUSE), desc="Mouse")]
        zebrafish_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_ZEBRAFISH), desc="Zebrafish")]
        chicken_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_CHICKEN), desc="Chicken")]
        background_seqs = [generate_background_sequence() for _ in tqdm(range(N_BACKGROUND), desc="Background")]
else:
    print("\nGenerating synthetic dataset...")
    human_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_HUMAN), desc="Human")]
    mouse_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_MOUSE), desc="Mouse")]
    zebrafish_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_ZEBRAFISH), desc="Zebrafish")]
    chicken_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_CHICKEN), desc="Chicken")]
    background_seqs = [generate_background_sequence() for _ in tqdm(range(N_BACKGROUND), desc="Background")]

print(f"\n✓ Dataset generated: {len(human_seqs) + len(mouse_seqs) + len(zebrafish_seqs) + len(chicken_seqs) + len(background_seqs):,} total sequences")


### 3.3 Prepare Dataset

In [ ]:
from sklearn.model_selection import train_test_split

print("Preparing dataset...")

# Combine all sequences
all_sequences = []
all_labels = []
species_labels = []

# Human (species_id=0)
all_sequences.extend(human_seqs)
all_labels.extend([1] * len(human_seqs))
species_labels.extend([0] * len(human_seqs))

# Mouse (species_id=1)
all_sequences.extend(mouse_seqs)
all_labels.extend([1] * len(mouse_seqs))
species_labels.extend([1] * len(mouse_seqs))

# Zebrafish (species_id=2)
all_sequences.extend(zebrafish_seqs)
all_labels.extend([1] * len(zebrafish_seqs))
species_labels.extend([2] * len(zebrafish_seqs))

# Chicken (species_id=3)
all_sequences.extend(chicken_seqs)
all_labels.extend([1] * len(chicken_seqs))
species_labels.extend([3] * len(chicken_seqs))

# Background (species_id=4)
all_sequences.extend(background_seqs)
all_labels.extend([0] * len(background_seqs))
species_labels.extend([4] * len(background_seqs))

print(f"Total sequences: {len(all_sequences):,}")

# One-hot encode
print("\nOne-hot encoding...")
X = np.array([one_hot_encode(seq) for seq in tqdm(all_sequences, desc="Encoding")])
y = np.array(all_labels, dtype=np.int64)
species = np.array(species_labels, dtype=np.int64)

print(f"Dataset shape: X={X.shape}, y={y.shape}")

# Create stratified splits
print("\nCreating train/val/test splits...")
indices = np.arange(len(y))

train_val_idx, test_idx = train_test_split(
    indices, test_size=0.15, stratify=y, random_state=42
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.176, stratify=y[train_val_idx], random_state=42
)

print(f"Train: {len(train_idx):,} ({len(train_idx)/len(y)*100:.1f}%)")
print(f"Val: {len(val_idx):,} ({len(val_idx)/len(y)*100:.1f}%)")
print(f"Test: {len(test_idx):,} ({len(test_idx)/len(y)*100:.1f}%)")

### 3.4 Save Dataset

In [ ]:
# Save processed data
data_dir = Path(CONFIG['data_dir'])
data_dir.mkdir(parents=True, exist_ok=True)

print("Saving dataset...")
np.savez_compressed(
    data_dir / 'dataset.npz',
    X=X, y=y, species=species
)

np.savez(
    data_dir / 'splits.npz',
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx
)

metadata = {
    'total_sequences': len(X),
    'sequence_length': 200,
    'n_train': len(train_idx),
    'n_val': len(val_idx),
    'n_test': len(test_idx),
    'species_names': ['human', 'mouse', 'zebrafish', 'chicken', 'background']
}

with open(data_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Dataset saved to {data_dir}")

## 4. Model Architecture

### 4.1 Define Model Components

In [ ]:
import math
import torch.nn.functional as F

# ===================================================================
# Model Architecture Components
# ===================================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for Transformer."""
    
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term[:pe[:, 0::2].shape[1]])
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class CNNEncoder(nn.Module):
    """CNN encoder supporting variable number of layers."""
    
    def __init__(self, input_channels=4, hidden_channels=None, kernel_sizes=None):
        super().__init__()
        
        if hidden_channels is None:
            hidden_channels = [128, 256, 384]
        if kernel_sizes is None:
            kernel_sizes = [7, 11, 15]
        
        self.hidden_channels = hidden_channels
        self.layers = nn.ModuleList()
        
        # Build layers dynamically to support variable number of layers
        in_ch = input_channels
        for out_ch, kernel in zip(hidden_channels, kernel_sizes):
            self.layers.append(nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel, padding=kernel//2),
                nn.BatchNorm1d(out_ch),
                nn.ReLU()
            ))
            in_ch = out_ch
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class HybridEnhancerModel(nn.Module):
    """Hybrid CNN-Transformer model for enhancer prediction."""
    
    def __init__(self, n_transformer_layers=10, n_attention_heads=8, d_model=256,
                 dim_feedforward=1536, dropout=0.1, cnn_channels=None, 
                 cnn_kernels=None, sequence_length=200, num_classes=2):
        super().__init__()
        
        if cnn_channels is None:
            cnn_channels = [128, 256, 384]
        if cnn_kernels is None:
            cnn_kernels = [7, 11, 15]
        
        self.d_model = d_model
        self.n_layers = n_transformer_layers
        self.n_heads = n_attention_heads
        
        # CNN Encoder
        self.cnn_encoder = CNNEncoder(4, cnn_channels, cnn_kernels)
        
        # Project CNN features to d_model
        self.projection = nn.Linear(cnn_channels[-1], d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, sequence_length, dropout)
        
        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, n_attention_heads, dim_feedforward, dropout,
            activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_transformer_layers)
        
        # Classification head
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 256)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x, return_features=False):
        # CNN encoding
        x = self.cnn_encoder(x)
        # Transpose and project
        x = x.transpose(1, 2)
        x = self.projection(x)
        # Positional encoding
        x = self.pos_encoder(x)
        # Transformer
        x = self.transformer(x)
        # Pool and classify
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)
    
    def get_num_params(self, trainable_only=True):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

print("✓ Model architecture classes defined")


In [ ]:
# Create model with GPU-specific configuration
model = HybridEnhancerModel(
    n_transformer_layers=CONFIG['n_transformer_layers'],
    n_attention_heads=CONFIG['n_attention_heads'],
    d_model=CONFIG['d_model'],
    dim_feedforward=CONFIG['dim_feedforward'],
    dropout=CONFIG['dropout'],
    cnn_channels=CONFIG['cnn_channels'],
    cnn_kernels=CONFIG['cnn_kernels']
).to(device)

n_params = model.get_num_params() / 1e6
print(f"\n{'='*70}")
print(f"MODEL ARCHITECTURE ({GPU_TYPE})")
print(f"{'='*70}")
print(f"Total parameters: {n_params:.1f}M")
print(f"  Transformer layers: {CONFIG['n_transformer_layers']}")
print(f"  Attention heads: {CONFIG['n_attention_heads']}")
print(f"  Model dimension: {CONFIG['d_model']}")
print(f"  Feedforward dim: {CONFIG['dim_feedforward']}")
print(f"\nExpected VRAM usage:")
if GPU_TYPE == 'A100':
    print(f"  Training: ~40-50GB (peak)")
    print(f"  Inference: ~20-25GB")
elif GPU_TYPE == 'V100':
    print(f"  Training: ~25-30GB (peak)")
    print(f"  Inference: ~12-15GB")
else:
    print(f"  Training: ~12-14GB (peak)")
    print(f"  Inference: ~6-8GB")
print(f"{'='*70}")

# Test forward pass
test_input = torch.randn(4, 4, 200).to(device)
with torch.no_grad():
    test_output = model(test_input)
print(f"\n✓ Test forward pass: input={test_input.shape}, output={test_output.shape}")
print(f"✓ Model ready for training")

## 5. Training Pipeline

### 5.1 Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

class EnhancerDataset(Dataset):
    def __init__(self, X, y, species=None):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.species = torch.from_numpy(species).long() if species is not None else None
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class StochasticEpochSampler:
    """Samples random subset each epoch for memory efficiency."""
    def __init__(self, train_indices, samples_per_epoch=100000, seed=42):
        self.train_indices = np.array(train_indices)
        self.samples_per_epoch = min(samples_per_epoch, len(self.train_indices))
        self.rng = np.random.RandomState(seed)
    
    def sample_epoch(self):
        return self.rng.choice(self.train_indices, size=self.samples_per_epoch, replace=False)


# Create datasets
full_dataset = EnhancerDataset(X, y, species)

# Validation loader
val_loader = DataLoader(
    full_dataset, batch_size=CONFIG['batch_size'],
    sampler=SubsetRandomSampler(val_idx),
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory']
)

# Test loader
test_loader = DataLoader(
    full_dataset, batch_size=CONFIG['batch_size'],
    sampler=SubsetRandomSampler(test_idx),
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory']
)

# Stochastic sampler for training
train_sampler = StochasticEpochSampler(
    train_idx, samples_per_epoch=CONFIG['samples_per_epoch']
)

print("Datasets created!")

### 5.2 Checkpointing Functions

In [ ]:
def save_checkpoint(state, checkpoint_dir, is_best=False):
    """Save training checkpoint."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True, parents=True)
    
    torch.save(state, checkpoint_dir / 'checkpoint_latest.pt')
    print(f"   Checkpoint saved: epoch {state['epoch']}")
    
    if is_best:
        torch.save(state, checkpoint_dir / 'checkpoint_best.pt')
        print(f"   Best model updated!")
    
    if state['epoch'] % 5 == 0:
        torch.save(state, checkpoint_dir / f"checkpoint_epoch_{state['epoch']}.pt")


def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None, scaler=None):
    """Load training checkpoint."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return None
    
    print(f"Loading checkpoint: {checkpoint_path.name}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if scaler and 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    
    print(f"   Resumed from epoch {checkpoint['epoch']}")
    return checkpoint

### 5.3 Training Loop

In [ ]:
import torch.optim as optim

def train_model(model, full_dataset, train_sampler, val_loader, device, config):
    """Complete training loop with checkpointing."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=config['scheduler_factor'],
        patience=config['scheduler_patience'], min_lr=config['scheduler_min_lr']
    )
    
    use_amp = config['use_mixed_precision']
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    # Try to resume
    start_epoch = 0
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    checkpoint = load_checkpoint(
        Path(config['checkpoint_dir']) / 'checkpoint_latest.pt',
        model, optimizer, scheduler, scaler
    )
    
    if checkpoint:
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        patience_counter = checkpoint['patience_counter']
        history = checkpoint['history']
    
    # Training loop
    for epoch in range(start_epoch, config['num_epochs']):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch+1}/{config['num_epochs']}")
        print(f"{'='*60}")
        
        # Sample for this epoch
        epoch_indices = train_sampler.sample_epoch()
        train_loader = DataLoader(
            full_dataset, batch_size=config['batch_size'],
            sampler=SubsetRandomSampler(epoch_indices),
            num_workers=config['num_workers'], pin_memory=config['pin_memory']
        )
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc="Training")
        for batch_idx, (sequences, labels) in enumerate(pbar):
            sequences, labels = sequences.to(device), labels.to(device)
            
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(sequences)
                loss = criterion(outputs, labels) / config['gradient_accumulation_steps']
            
            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            
            if (batch_idx + 1) % config['gradient_accumulation_steps'] == 0:
                if scaler:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                
                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * config['gradient_accumulation_steps']
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': f'{loss.item() * config["gradient_accumulation_steps"]:.4f}'})
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for sequences, labels in tqdm(val_loader, desc="Validation"):
                sequences, labels = sequences.to(device), labels.to(device)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(sequences)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        scheduler.step(avg_val_loss)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"   Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        
        is_best = avg_val_loss < best_val_loss
        if is_best:
            best_val_loss = avg_val_loss
            patience_counter = 0
            print("   New best model!")
        else:
            patience_counter += 1
            print(f"   No improvement ({patience_counter}/{config['early_stopping_patience']})")
        
        # Save checkpoint
        save_checkpoint({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'best_val_loss': best_val_loss,
            'patience_counter': patience_counter,
            'history': history,
        }, config['checkpoint_dir'], is_best=is_best)
        
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Load best model
    load_checkpoint(Path(config['checkpoint_dir']) / 'checkpoint_best.pt', model)
    
    return history

### 5.4 Run Training

In [ ]:
# Train the model
print("Starting training...")
print(f"Checkpoints will be saved to: {CONFIG['checkpoint_dir']}")

history = train_model(
    model=model,
    full_dataset=full_dataset,
    train_sampler=train_sampler,
    val_loader=val_loader,
    device=device,
    config=CONFIG
)

print("\nTraining complete!")

### 5.5 Plot Training Curves

In [ ]:
# Plot training curves with publication quality (300 DPI)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train', marker='o', markersize=4)
axes[0].plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Validation', marker='s', markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history['train_acc'], 'b-', linewidth=2, label='Train', marker='o', markersize=4)
axes[1].plot(epochs, history['val_acc'], 'r-', linewidth=2, label='Validation', marker='s', markersize=4)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
# Save at 300 DPI for publication quality
plt.savefig(f"{CONFIG['figures_dir']}/training_curves.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nTraining Summary:")
print(f"  Final Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Final Train Acc:  {history['train_acc'][-1]:.2f}%")
print(f"  Final Val Loss:   {history['val_loss'][-1]:.4f}")
print(f"  Final Val Acc:    {history['val_acc'][-1]:.2f}%")
print(f"\n✓ Figure saved at 300 DPI: {CONFIG['figures_dir']}/training_curves.png")

### 5.6 Evaluate on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_recall_fscore_support
from scipy import stats

model.eval()
test_loss = 0.0
all_preds = []
all_labels = []
all_probs = []

criterion = nn.CrossEntropyLoss()

print("Evaluating on test set...")
with torch.no_grad():
    for sequences, labels in tqdm(test_loader, desc="Testing"):
        sequences, labels = sequences.to(device), labels.to(device)
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

test_loss /= len(test_loader)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = 100. * (all_preds == all_labels).sum() / len(all_labels)
test_auc = roc_auc_score(all_labels, all_probs)
cm = confusion_matrix(all_labels, all_preds)

# Calculate precision, recall, F1 with confidence intervals
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, average='binary'
)

# Bootstrap confidence intervals (95%)
def bootstrap_ci(labels, preds, metric_func, n_bootstrap=1000, ci=95):
    """Calculate bootstrap confidence interval."""
    np.random.seed(42)
    scores = []
    n = len(labels)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, n, replace=True)
        score = metric_func(labels[indices], preds[indices])
        scores.append(score)
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    return lower, upper

acc_ci = bootstrap_ci(all_labels, all_preds, lambda y, p: (y == p).mean() * 100)
f1_ci = bootstrap_ci(all_labels, all_preds, 
    lambda y, p: precision_recall_fscore_support(y, p, average='binary')[2])

print(f"\n{'='*70}")
print(f"TEST SET RESULTS")
print(f"{'='*70}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}% (95% CI: [{acc_ci[0]:.2f}%, {acc_ci[1]:.2f}%])")
print(f"ROC AUC: {test_auc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f} (95% CI: [{f1_ci[0]:.3f}, {f1_ci[1]:.3f}])")

print(f"\nConfusion Matrix:")
tn, fp, fn, tp = cm.ravel()
print(f"                Predicted")
print(f"              Neg      Pos")
print(f"Actual  Neg   {tn:6,}   {fp:6,}")
print(f"        Pos   {fn:6,}   {tp:6,}")

# Calculate per-class metrics
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
print(f"\nPer-Class Metrics:")
print(f"  Sensitivity (Recall): {recall:.3f}")
print(f"  Specificity: {specificity:.3f}")
print(f"  Positive Predictive Value (Precision): {precision:.3f}")
print(f"  Negative Predictive Value: {tn/(tn+fn) if (tn+fn)>0 else 0:.3f}")
print(f"{'='*70}")

test_results = {
    'test_loss': test_loss,
    'test_accuracy': test_acc,
    'accuracy_ci': acc_ci,
    'auc': test_auc,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'f1_ci': f1_ci,
    'confusion_matrix': cm,
    'specificity': specificity,
    'predictions': all_preds,
    'labels': all_labels,
    'probabilities': all_probs
}

print("\n✓ Test evaluation complete with confidence intervals")

## 6. Innovation #1: Cross-Species Evolutionary Analysis

Test repair success across vertebrate species.

In [ ]:
def damage_sequence(sequence, damage_fraction=0.1, seed=None):
    """Randomly damage DNA sequence."""
    if seed is not None:
        np.random.seed(seed)
    
    sequence = sequence.copy()
    seq_length = sequence.shape[1]
    n_damage = int(seq_length * damage_fraction)
    damaged_positions = np.random.choice(seq_length, size=n_damage, replace=False)
    
    for pos in damaged_positions:
        current_nt = np.argmax(sequence[:, pos])
        possible = [0, 1, 2, 3]
        possible.remove(current_nt)
        new_nt = np.random.choice(possible)
        sequence[:, pos] = 0
        sequence[new_nt, pos] = 1
    
    return sequence, damaged_positions


def attempt_repair(damaged_sequence, model, device, max_iterations=30, 
                   learning_rate=0.01, target_prob=0.9):
    """Attempt to repair damaged sequence using gradient descent."""
    model.eval()
    
    seq_tensor = torch.from_numpy(damaged_sequence).float().unsqueeze(0).to(device)
    seq_tensor.requires_grad_(True)
    
    trajectory = []
    
    for iteration in range(max_iterations):
        outputs = model(seq_tensor)
        probs = torch.softmax(outputs, dim=1)
        enhancer_prob = probs[0, 1]
        
        trajectory.append(enhancer_prob.item())
        
        if enhancer_prob.item() >= target_prob:
            return True, enhancer_prob.item(), trajectory
        
        loss = -torch.log(enhancer_prob + 1e-8)
        loss.backward()
        
        with torch.no_grad():
            seq_tensor.data -= learning_rate * seq_tensor.grad
            seq_data = seq_tensor.squeeze(0)
            max_indices = torch.argmax(seq_data, dim=0)
            new_seq = torch.zeros_like(seq_data)
            for pos in range(200):
                new_seq[max_indices[pos], pos] = 1.0
            seq_tensor.data = new_seq.unsqueeze(0)
            if seq_tensor.grad is not None:
                seq_tensor.grad.zero_()
    
    return False, trajectory[-1] if trajectory else 0.0, trajectory

In [ ]:
# Run cross-species analysis
print("Running cross-species repair analysis...")

species_names = ['Human', 'Mouse', 'Zebrafish', 'Chicken']
cross_species_results = {}

for species_id, species_name in enumerate(species_names):
    # Get test enhancers for this species
    species_test_mask = (species[test_idx] == species_id) & (y[test_idx] == 1)
    species_seqs = X[test_idx][species_test_mask]
    
    if len(species_seqs) == 0:
        print(f"No test sequences for {species_name}")
        continue
    
    n_test = min(CROSS_SPECIES_CONFIG['n_sequences_per_species'], len(species_seqs))
    test_seqs = species_seqs[:n_test]
    
    successes = []
    print(f"\n{species_name}: Testing {n_test} sequences...")
    
    for seq in tqdm(test_seqs, desc=f"  {species_name}"):
        damaged, _ = damage_sequence(seq, damage_fraction=CROSS_SPECIES_CONFIG['damage_fraction'])
        success, _, _ = attempt_repair(
            damaged, model, device,
            max_iterations=CROSS_SPECIES_CONFIG['max_repair_iterations'],
            learning_rate=CROSS_SPECIES_CONFIG['repair_learning_rate'],
            target_prob=CROSS_SPECIES_CONFIG['target_probability']
        )
        successes.append(success)
    
    repair_rate = np.mean(successes) * 100
    cross_species_results[species_name] = {
        'repair_rate': repair_rate,
        'n_tested': n_test,
        'successes': sum(successes)
    }
    print(f"  Repair rate: {repair_rate:.1f}%")

print("\nCross-species analysis complete!")

In [ ]:
# Visualize cross-species results with statistical tests
from scipy import stats as scipy_stats

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Repair rates
ax = axes[0]
species = list(cross_species_results.keys())
repair_rates = [cross_species_results[s]['repair_rate'] for s in species]
n_tested = [cross_species_results[s]['n_tested'] for s in species]
successes = [cross_species_results[s]['successes'] for s in species]

# Calculate confidence intervals (binomial)
cis = []
for n_succ, n_tot in zip(successes, n_tested):
    p = n_succ / n_tot if n_tot > 0 else 0
    # Wilson score interval
    z = 1.96  # 95% CI
    denominator = 1 + z**2 / n_tot
    center = (p + z**2 / (2 * n_tot)) / denominator
    margin = z * np.sqrt(p * (1 - p) / n_tot + z**2 / (4 * n_tot**2)) / denominator
    ci_low = max(0, (center - margin) * 100)
    ci_high = min(100, (center + margin) * 100)
    cis.append((ci_high - p * 100, p * 100 - ci_low))  # Error bars

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(species)]
errors = np.array(cis).T

bars = ax.bar(species, repair_rates, yerr=errors, capsize=5,
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', 
            fontsize=12, fontweight='bold')

ax.set_ylabel('Repair Success Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Cross-Species Repair Analysis', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xlabel('Species', fontsize=13, fontweight='bold')

# Panel B: Statistical comparison
ax = axes[1]

# Pairwise chi-square tests
if len(species) >= 2:
    # Compare each species to human (baseline)
    human_idx = species.index('Human') if 'Human' in species else 0
    human_succ = successes[human_idx]
    human_total = n_tested[human_idx]
    
    pvalues = []
    comparisons = []
    
    for j, sp in enumerate(species):
        if j != human_idx:
            # Chi-square test
            contingency = [
                [successes[human_idx], n_tested[human_idx] - successes[human_idx]],
                [successes[j], n_tested[j] - successes[j]]
            ]
            chi2, pval = scipy_stats.chi2_contingency(contingency)[:2]
            pvalues.append(pval)
            comparisons.append(f'Human vs\n{sp}')
    
    # Plot p-values
    y_pos = np.arange(len(comparisons))
    bars = ax.barh(y_pos, [-np.log10(p) for p in pvalues],
                   color=['red' if p < 0.05 else 'gray' for p in pvalues],
                   alpha=0.7, edgecolor='black')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(comparisons, fontsize=10)
    ax.set_xlabel('-log10(p-value)', fontsize=12, fontweight='bold')
    ax.set_title('Statistical Significance\n(Chi-square test)', 
                fontsize=13, fontweight='bold')
    ax.axvline(-np.log10(0.05), color='black', linestyle='--', 
              linewidth=2, label='p=0.05')
    ax.axvline(-np.log10(0.01), color='black', linestyle=':', 
              linewidth=2, label='p=0.01')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add p-value labels
    for i, (bar, pval) in enumerate(zip(bars, pvalues)):
        width = bar.get_width()
        label = f'p={pval:.3f}' if pval >= 0.001 else 'p<0.001'
        ax.text(width, bar.get_y() + bar.get_height()/2., f'  {label}',
                ha='left', va='center', fontsize=9)

plt.tight_layout()
# Save at 300 DPI for publication quality
plt.savefig(f"{CONFIG['figures_dir']}/cross_species_repair.png", 
           dpi=300, bbox_inches='tight')
plt.show()

# Save statistical results
stats_results = {
    'species': species,
    'repair_rates': repair_rates,
    'confidence_intervals': cis,
    'n_tested': n_tested,
    'successes': successes,
}

if len(species) >= 2:
    stats_results['pairwise_pvalues'] = dict(zip(comparisons, pvalues))

with open(f"{CONFIG['results_dir']}/cross_species/results.pkl", 'wb') as f:
    pickle.dump(cross_species_results, f)

with open(f"{CONFIG['results_dir']}/cross_species/statistics.pkl", 'wb') as f:
    pickle.dump(stats_results, f)

print("\n✓ Cross-species analysis complete with statistical tests")
print(f"✓ Figure saved at 300 DPI: {CONFIG['figures_dir']}/cross_species_repair.png")

## 7. Innovation #2: Hierarchical Grammar Discovery

Discover which motifs work synergistically vs independently.

In [ ]:
# Extract important regions using gradient importance
print("Extracting motif regions...")

# Get enhancer test sequences
enhancer_test_mask = y[test_idx] == 1
enhancer_seqs = X[test_idx][enhancer_test_mask][:GRAMMAR_CONFIG['n_sequences_for_motifs']]

all_importances = []

for seq in tqdm(enhancer_seqs, desc="Computing importance"):
    seq_tensor = torch.from_numpy(seq).float().unsqueeze(0).to(device)
    seq_tensor.requires_grad_(True)
    
    outputs = model(seq_tensor)
    enhancer_prob = torch.softmax(outputs, dim=1)[0, 1]
    enhancer_prob.backward()
    
    grad = seq_tensor.grad.abs().sum(dim=1).squeeze().detach().cpu().numpy()
    all_importances.append(grad)

avg_importance = np.mean(all_importances, axis=0)

# Find peaks
from scipy.signal import find_peaks
threshold = np.percentile(avg_importance, GRAMMAR_CONFIG['importance_percentile'])
peaks, _ = find_peaks(avg_importance, height=threshold, distance=6)

# Define motif regions
motif_regions = [(max(0, p-4), min(200, p+5)) for p in peaks]
print(f"Found {len(motif_regions)} candidate motif regions")

In [ ]:
# Test pairwise interactions
def damage_region(sequence, start, end):
    """Damage specific region."""
    damaged = sequence.copy()
    for pos in range(start, min(end, sequence.shape[1])):
        current = np.argmax(sequence[:, pos])
        possible = [0, 1, 2, 3]
        possible.remove(current)
        damaged[:, pos] = 0
        damaged[np.random.choice(possible), pos] = 1
    return damaged

print("Discovering grammar rules...")
interactions = []
n_motifs = len(motif_regions)

if n_motifs >= 2:
    pairs_tested = set()
    
    for _ in tqdm(range(min(GRAMMAR_CONFIG['n_pairs_to_test'], n_motifs*(n_motifs-1)//2)), desc="Testing pairs"):
        # Get unique pair
        attempts = 0
        while attempts < 100:
            i, j = np.random.choice(n_motifs, size=2, replace=False)
            pair_key = (min(i, j), max(i, j))
            if pair_key not in pairs_tested:
                pairs_tested.add(pair_key)
                break
            attempts += 1
        
        if attempts >= 100:
            break
        
        motif1, motif2 = motif_regions[i], motif_regions[j]
        
        results = []
        for seq_idx in range(min(GRAMMAR_CONFIG['n_sequences_per_pair'], len(enhancer_seqs))):
            seq = enhancer_seqs[seq_idx]
            
            dam1 = damage_region(seq, motif1[0], motif1[1])
            r1, _, _ = attempt_repair(dam1, model, device)
            
            dam2 = damage_region(seq, motif2[0], motif2[1])
            r2, _, _ = attempt_repair(dam2, model, device)
            
            dam_both = damage_region(dam1, motif2[0], motif2[1])
            r_both, _, _ = attempt_repair(dam_both, model, device)
            
            results.append((r1, r2, r_both))
        
        avg_r1 = np.mean([r[0] for r in results])
        avg_r2 = np.mean([r[1] for r in results])
        avg_r_both = np.mean([r[2] for r in results])
        expected = avg_r1 * avg_r2
        
        if expected > 0:
            ratio = avg_r_both / expected
        else:
            ratio = 1.0
        
        if ratio < 0.5:
            interaction_type = "SYNERGY"
        elif ratio > 1.5:
            interaction_type = "REDUNDANT"
        else:
            interaction_type = "INDEPENDENT"
        
        interactions.append({
            'motif1': i, 'motif2': j,
            'repair1': avg_r1, 'repair2': avg_r2, 'repair_both': avg_r_both,
            'interaction': interaction_type
        })

# Summarize
synergistic = [x for x in interactions if x['interaction'] == 'SYNERGY']
redundant = [x for x in interactions if x['interaction'] == 'REDUNDANT']
independent = [x for x in interactions if x['interaction'] == 'INDEPENDENT']

print(f"\nGrammar Discovery Results:")
print(f"   Synergistic: {len(synergistic)} ({len(synergistic)/max(1,len(interactions))*100:.1f}%)")
print(f"   Redundant: {len(redundant)} ({len(redundant)/max(1,len(interactions))*100:.1f}%)")
print(f"   Independent: {len(independent)} ({len(independent)/max(1,len(interactions))*100:.1f}%)")

In [ ]:
# Visualize grammar results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Importance profile
ax = axes[0]
ax.plot(avg_importance, linewidth=2, color='darkblue')
ax.fill_between(range(200), 0, avg_importance, alpha=0.3, color='skyblue')
for start, end in motif_regions:
    ax.axvspan(start, end, alpha=0.2, color='red')
ax.set_xlabel('Position (bp)')
ax.set_ylabel('Importance')
ax.set_title('Position Importance Profile', fontweight='bold')
ax.grid(True, alpha=0.3)

# Interaction pie chart
ax = axes[1]
counts = [len(synergistic), len(redundant), len(independent)]
if sum(counts) > 0:
    ax.pie(counts, labels=['Synergistic', 'Redundant', 'Independent'], 
           autopct='%1.1f%%', colors=['#e74c3c', '#f39c12', '#95a5a6'])
ax.set_title('Grammar Rule Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}/grammar_discovery.png", dpi=300, bbox_inches='tight')
plt.show()

# Save results
with open(f"{CONFIG['results_dir']}/grammar/results.pkl", 'wb') as f:
    pickle.dump({'motif_regions': motif_regions, 'interactions': interactions}, f)

## 8. Innovation #3: Generative Validation

Generate functional enhancers from random DNA.

In [ ]:
def generate_enhancer_from_random(model, device, max_iterations=500, target_prob=0.80, lr=0.05):
    """Generate enhancer from random DNA."""
    model.eval()
    
    # Random initial sequence
    random_seq = np.zeros((4, 200), dtype=np.float32)
    for i in range(200):
        random_seq[np.random.randint(0, 4), i] = 1.0
    
    seq_tensor = torch.from_numpy(random_seq).float().unsqueeze(0).to(device)
    seq_tensor.requires_grad_(True)
    
    trajectory = []
    
    for _ in range(max_iterations):
        outputs = model(seq_tensor)
        probs = torch.softmax(outputs, dim=1)
        enhancer_prob = probs[0, 1]
        trajectory.append(enhancer_prob.item())
        
        if enhancer_prob.item() >= target_prob:
            return seq_tensor.squeeze(0).detach().cpu().numpy(), trajectory, True
        
        loss = -torch.log(enhancer_prob + 1e-8)
        loss.backward()
        
        with torch.no_grad():
            seq_tensor.data -= lr * seq_tensor.grad
            seq_data = seq_tensor.squeeze(0)
            max_indices = torch.argmax(seq_data, dim=0)
            new_seq = torch.zeros_like(seq_data)
            for pos in range(200):
                new_seq[max_indices[pos], pos] = 1.0
            seq_tensor.data = new_seq.unsqueeze(0)
            if seq_tensor.grad is not None:
                seq_tensor.grad.zero_()
    
    return seq_tensor.squeeze(0).detach().cpu().numpy(), trajectory, False


# Generate enhancers
print("Generating synthetic enhancers...")

generated_seqs = []
trajectories = []

for _ in tqdm(range(GENERATIVE_CONFIG['n_generation_attempts']), desc="Generating"):
    seq, traj, success = generate_enhancer_from_random(
        model, device,
        max_iterations=GENERATIVE_CONFIG['max_iterations'],
        target_prob=GENERATIVE_CONFIG['target_probability'],
        lr=GENERATIVE_CONFIG['learning_rate']
    )
    if success:
        generated_seqs.append(seq)
        trajectories.append(traj)

success_rate = len(generated_seqs) / GENERATIVE_CONFIG['n_generation_attempts'] * 100
print(f"\nGenerated {len(generated_seqs)} enhancers ({success_rate:.1f}% success rate)")

In [ ]:
# Validate generated sequences with comprehensive statistical tests
from scipy import stats

known_motifs = ['TATAAA', 'CACGTG', 'GGGCGG', 'CCAAT', 'GCGCGC']

def calc_gc(seq_str):
    return (seq_str.count('G') + seq_str.count('C')) / len(seq_str)

def count_motifs(seq_str, motifs):
    return sum(seq_str.count(m) for m in motifs)

# Analyze generated sequences
if len(generated_seqs) > 0:
    gen_sequences_str = [decode_sequence(s) for s in generated_seqs]
    gen_gc = [calc_gc(s) for s in gen_sequences_str]
    gen_motifs = [count_motifs(s, known_motifs) for s in gen_sequences_str]
    
    # Analyze real sequences
    real_seqs = X[test_idx][y[test_idx] == 1][:1000]
    real_sequences_str = [decode_sequence(s) for s in real_seqs]
    real_gc = [calc_gc(s) for s in real_sequences_str]
    real_motifs = [count_motifs(s, known_motifs) for s in real_sequences_str]
    
    # Statistical tests
    # 1. Kolmogorov-Smirnov test for GC distribution
    ks_stat_gc, ks_pvalue_gc = stats.ks_2samp(gen_gc, real_gc)
    
    # 2. T-test for GC content (comparing means)
    t_stat_gc, t_pvalue_gc = stats.ttest_ind(gen_gc, real_gc)
    
    # 3. Mann-Whitney U test for motif counts (non-parametric)
    u_stat_motif, mw_pvalue_motif = stats.mannwhitneyu(gen_motifs, real_motifs, alternative='two-sided')
    
    # 4. T-test for motif counts
    t_stat_motif, t_pvalue_motif = stats.ttest_ind(gen_motifs, real_motifs)
    
    print(f"\n{'='*70}")
    print(f"GENERATIVE VALIDATION RESULTS")
    print(f"{'='*70}")
    print(f"\nGeneration Statistics:")
    print(f"  Attempts: {GENERATIVE_CONFIG['n_generation_attempts']}")
    print(f"  Successes: {len(generated_seqs)}")
    print(f"  Success Rate: {success_rate:.1f}%")
    
    print(f"\nGC Content Comparison:")
    print(f"  Generated: {np.mean(gen_gc):.4f} ± {np.std(gen_gc):.4f}")
    print(f"  Real:      {np.mean(real_gc):.4f} ± {np.std(real_gc):.4f}")
    print(f"  T-test p-value: {t_pvalue_gc:.4f} {'✓ Similar' if t_pvalue_gc > 0.05 else '✗ Different'}")
    print(f"  KS-test p-value: {ks_pvalue_gc:.4f} {'✓ Similar distribution' if ks_pvalue_gc > 0.05 else '✗ Different distribution'}")
    print(f"  KS statistic: {ks_stat_gc:.4f}")
    
    print(f"\nMotif Content Comparison:")
    print(f"  Generated: {np.mean(gen_motifs):.2f} ± {np.std(gen_motifs):.2f} motifs/sequence")
    print(f"  Real:      {np.mean(real_motifs):.2f} ± {np.std(real_motifs):.2f} motifs/sequence")
    print(f"  T-test p-value: {t_pvalue_motif:.4f} {'✓ Similar' if t_pvalue_motif > 0.05 else '✗ Different'}")
    print(f"  Mann-Whitney p-value: {mw_pvalue_motif:.4f} {'✓ Similar' if mw_pvalue_motif > 0.05 else '✗ Different'}")
    
    # Hypothesis testing summary
    print(f"\n{'='*70}")
    print(f"HYPOTHESIS TESTING SUMMARY")
    print(f"{'='*70}")
    print(f"H0: Generated sequences are indistinguishable from real enhancers")
    print(f"  GC distribution: {'ACCEPT H0' if ks_pvalue_gc > 0.05 else 'REJECT H0'} (α=0.05)")
    print(f"  Motif content:   {'ACCEPT H0' if mw_pvalue_motif > 0.05 else 'REJECT H0'} (α=0.05)")
    
    overall_valid = (ks_pvalue_gc > 0.05) and (mw_pvalue_motif > 0.05)
    print(f"\nOverall validation: {'✓ PASSED' if overall_valid else '✗ FAILED'}")
    if overall_valid:
        print("  Model demonstrates causal understanding of enhancer grammar")
    else:
        print("  Model needs improvement - generated sequences differ from real")
    print(f"{'='*70}")
    
    validation_results = {
        'generated_gc': gen_gc,
        'real_gc': real_gc,
        'generated_motifs': gen_motifs,
        'real_motifs': real_motifs,
        'success_rate': success_rate,
        'n_generated': len(generated_seqs),
        'n_attempts': GENERATIVE_CONFIG['n_generation_attempts'],
        'statistical_tests': {
            'gc_ttest_pvalue': t_pvalue_gc,
            'gc_kstest_pvalue': ks_pvalue_gc,
            'gc_kstest_statistic': ks_stat_gc,
            'motif_ttest_pvalue': t_pvalue_motif,
            'motif_mwtest_pvalue': mw_pvalue_motif,
            'motif_mwtest_statistic': u_stat_motif,
        },
        'validation_passed': overall_valid
    }
else:
    print("\n⚠️  No sequences generated - cannot validate")
    validation_results = {'success_rate': 0, 'validation_passed': False}

print("\n✓ Generative validation complete with statistical tests")

In [ ]:
# Visualize generative results with statistical comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Generation trajectories
ax = axes[0]
for traj in trajectories[:50]:  # Show more trajectories
    ax.plot(traj, alpha=0.15, color='blue', linewidth=0.8)
if trajectories:
    max_len = max(len(t) for t in trajectories)
    avg_traj = [np.mean([t[i] for t in trajectories if len(t) > i]) for i in range(max_len)]
    ax.plot(avg_traj, color='red', linewidth=3, label='Average', zorder=10)
ax.axhline(y=0.80, color='green', linestyle='--', linewidth=2, label='Target (0.80)', zorder=5)
ax.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax.set_ylabel('Enhancer Probability', fontsize=12, fontweight='bold')
ax.set_title('Generation Trajectories', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Panel B: GC content comparison with KS test
ax = axes[1]
if len(gen_gc) > 0 and len(real_gc) > 0:
    # Histograms
    bins = np.linspace(0.3, 0.6, 31)
    ax.hist(real_gc, bins=bins, alpha=0.6, label='Real Enhancers', 
            color='blue', edgecolor='black', density=True)
    ax.hist(gen_gc, bins=bins, alpha=0.6, label='Generated', 
            color='red', edgecolor='black', density=True)
    
    # Add vertical lines for means
    ax.axvline(np.mean(real_gc), color='blue', linestyle='--', 
              linewidth=2, alpha=0.8, label=f'Real mean: {np.mean(real_gc):.3f}')
    ax.axvline(np.mean(gen_gc), color='red', linestyle='--', 
              linewidth=2, alpha=0.8, label=f'Gen mean: {np.mean(gen_gc):.3f}')
    
    # Add KS test result
    if 'gc_kstest_pvalue' in validation_results.get('statistical_tests', {}):
        ks_p = validation_results['statistical_tests']['gc_kstest_pvalue']
        ax.text(0.95, 0.95, f'KS test\np = {ks_p:.3f}',
                transform=ax.transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
                fontsize=10, fontweight='bold')
    
    ax.legend(fontsize=10)
ax.set_xlabel('GC Content', fontsize=12, fontweight='bold')
ax.set_ylabel('Density', fontsize=12, fontweight='bold')
ax.set_title('GC Content Distribution', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Panel C: Motif content comparison
ax = axes[2]
if len(gen_motifs) > 0 and len(real_motifs) > 0:
    # Box plots
    bp = ax.boxplot([real_motifs, gen_motifs], 
                    labels=['Real\nEnhancers', 'Generated'],
                    patch_artist=True,
                    widths=0.6)
    
    # Color the boxes
    colors = ['skyblue', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add means as points
    means = [np.mean(real_motifs), np.mean(gen_motifs)]
    ax.plot([1, 2], means, 'ro', markersize=10, label='Mean', zorder=10)
    
    # Add statistical test result
    if 'motif_mwtest_pvalue' in validation_results.get('statistical_tests', {}):
        mw_p = validation_results['statistical_tests']['motif_mwtest_pvalue']
        ax.text(0.5, 0.95, f'Mann-Whitney U\np = {mw_p:.3f}',
                transform=ax.transAxes, ha='center', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
                fontsize=10, fontweight='bold')
    
    ax.legend(fontsize=10)
ax.set_ylabel('Motif Count per Sequence', fontsize=12, fontweight='bold')
ax.set_title('Motif Content Comparison', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
# Save at 300 DPI for publication quality
plt.savefig(f"{CONFIG['figures_dir']}/generative_validation.png", 
           dpi=300, bbox_inches='tight')
plt.show()

# Save results
with open(f"{CONFIG['results_dir']}/generative/results.pkl", 'wb') as f:
    pickle.dump({
        'generated_seqs': generated_seqs, 
        'trajectories': trajectories, 
        'validation': validation_results
    }, f)

print("\n✓ Generative validation visualizations complete")
print(f"✓ Figure saved at 300 DPI: {CONFIG['figures_dir']}/generative_validation.png")

## 9. Comprehensive Summary Figure

In [ ]:
# Create comprehensive summary figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Panel 1: Training curves
ax1 = fig.add_subplot(gs[0, 0:2])
epochs = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train')
ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Progress', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Performance
ax2 = fig.add_subplot(gs[0, 2])
metrics = ['Train', 'Val', 'Test']
values = [history['train_acc'][-1], history['val_acc'][-1], test_results['test_accuracy']]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax2.bar(metrics, values, color=colors, alpha=0.8, edgecolor='black')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{bar.get_height():.1f}%', ha='center', va='bottom')
ax2.set_ylim(0, 100)
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Model Performance', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Panel 3: Cross-species
ax3 = fig.add_subplot(gs[1, 0])
species_list = list(cross_species_results.keys())
rates = [cross_species_results[s]['repair_rate'] for s in species_list]
ax3.bar(species_list, rates, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(species_list)], 
        alpha=0.8, edgecolor='black')
ax3.set_ylabel('Repair Rate (%)')
ax3.set_title('Cross-Species', fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3, axis='y')

# Panel 4: Grammar
ax4 = fig.add_subplot(gs[1, 1])
counts = [len(synergistic), len(redundant), len(independent)]
if sum(counts) > 0:
    ax4.pie(counts, labels=['Synergy', 'Redundant', 'Indep'], autopct='%1.1f%%',
            colors=['#e74c3c', '#f39c12', '#95a5a6'])
ax4.set_title('Grammar Rules', fontweight='bold')

# Panel 5: Generation
ax5 = fig.add_subplot(gs[1, 2])
gen_metrics = ['Success', 'GC Match', 'Motif Match']
gen_values = [
    validation_results.get('success_rate', 0),
    100 if validation_results.get('gc_pvalue', 0) > 0.05 else 50,
    100 if validation_results.get('motif_pvalue', 0) > 0.05 else 50
]
ax5.bar(gen_metrics, gen_values, color=['#9b59b6', '#1abc9c', '#e67e22'], alpha=0.8, edgecolor='black')
ax5.set_ylim(0, 100)
ax5.set_ylabel('Score')
ax5.set_title('Generation', fontweight='bold')
ax5.tick_params(axis='x', rotation=45)
ax5.grid(True, alpha=0.3, axis='y')

# Panel 6: Summary text
ax6 = fig.add_subplot(gs[2, :])
ax6.axis('off')
summary_text = f"""
RESEARCH SUMMARY
================
Model: Hybrid CNN-Transformer ({model.get_num_params()/1e6:.1f}M parameters)
Test Accuracy: {test_results['test_accuracy']:.1f}%  |  ROC AUC: {test_results['auc']:.3f}

Innovation #1 - Cross-Species Conservation:
  Tested repair across {len(cross_species_results)} vertebrate species
  Average repair rate: {np.mean([r['repair_rate'] for r in cross_species_results.values()]):.1f}%

Innovation #2 - Grammar Discovery:
  Found {len(motif_regions)} motif regions, tested {len(interactions)} pairs
  Synergistic: {len(synergistic)}  |  Redundant: {len(redundant)}  |  Independent: {len(independent)}

Innovation #3 - Generative Validation:
  Generation success rate: {validation_results.get('success_rate', 0):.1f}%
  GC content similar to real enhancers: {'Yes' if validation_results.get('gc_pvalue', 0) > 0.05 else 'No'}
"""
ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11, 
         verticalalignment='top', family='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('ENHANCER REPAIR RESEARCH: COMPREHENSIVE ANALYSIS', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(f"{CONFIG['figures_dir']}/comprehensive_summary.png", dpi=300, bbox_inches='tight')
plt.show()

## 11. Publication-Ready Results Tables

Generate comprehensive tables for publication.

In [ ]:
# Create publication-ready results tables
import pandas as pd

print("\n" + "="*70)
print("GENERATING PUBLICATION-READY TABLES")
print("="*70)

# Table 1: Model Performance Metrics
print("\nTable 1: Model Performance Metrics\n")
perf_table = pd.DataFrame({
    'Dataset': ['Training', 'Validation', 'Test'],
    'Loss': [
        f"{history['train_loss'][-1]:.4f}",
        f"{history['val_loss'][-1]:.4f}",
        f"{test_results['test_loss']:.4f}"
    ],
    'Accuracy (%)': [
        f"{history['train_acc'][-1]:.2f}",
        f"{history['val_acc'][-1]:.2f}",
        f"{test_results['test_accuracy']:.2f}"
    ],
    '95% CI': [
        '-',
        '-',
        f"[{test_results['accuracy_ci'][0]:.2f}, {test_results['accuracy_ci'][1]:.2f}]"
    ]
})
print(perf_table.to_string(index=False))

# Save to CSV
perf_table.to_csv(f"{CONFIG['results_dir']}/tables/model_performance.csv", index=False)

# Table 2: Test Set Detailed Metrics
print("\n\nTable 2: Test Set Detailed Metrics\n")
detailed_table = pd.DataFrame({
    'Metric': ['ROC AUC', 'Precision', 'Recall', 'F1 Score', 'Specificity'],
    'Value': [
        f"{test_results['auc']:.3f}",
        f"{test_results['precision']:.3f}",
        f"{test_results['recall']:.3f}",
        f"{test_results['f1']:.3f}",
        f"{test_results['specificity']:.3f}"
    ],
    '95% CI': [
        '-',
        '-',
        '-',
        f"[{test_results['f1_ci'][0]:.3f}, {test_results['f1_ci'][1]:.3f}]",
        '-'
    ]
})
print(detailed_table.to_string(index=False))
detailed_table.to_csv(f"{CONFIG['results_dir']}/tables/test_metrics.csv", index=False)

# Table 3: Cross-Species Analysis
print("\n\nTable 3: Cross-Species Repair Analysis\n")
species_data = []
for sp in cross_species_results.keys():
    r = cross_species_results[sp]
    species_data.append({
        'Species': sp,
        'N Tested': r['n_tested'],
        'Successes': r['successes'],
        'Repair Rate (%)': f"{r['repair_rate']:.1f}",
        'Evolutionary Distance': {
            'Human': '0 Mya',
            'Mouse': '~90 Mya',
            'Chicken': '~310 Mya',
            'Zebrafish': '~430 Mya'
        }.get(sp, 'N/A')
    })
cross_species_table = pd.DataFrame(species_data)
print(cross_species_table.to_string(index=False))
cross_species_table.to_csv(f"{CONFIG['results_dir']}/tables/cross_species.csv", index=False)

# Table 4: Grammar Discovery Summary
print("\n\nTable 4: Grammar Discovery Summary\n")
grammar_table = pd.DataFrame({
    'Interaction Type': ['Synergistic', 'Redundant', 'Independent'],
    'Count': [len(synergistic), len(redundant), len(independent)],
    'Percentage': [
        f"{len(synergistic)/max(1,len(interactions))*100:.1f}%" if len(interactions) > 0 else '0.0%',
        f"{len(redundant)/max(1,len(interactions))*100:.1f}%" if len(interactions) > 0 else '0.0%',
        f"{len(independent)/max(1,len(interactions))*100:.1f}%" if len(interactions) > 0 else '0.0%'
    ],
    'Biological Interpretation': [
        'Both motifs required for function',
        'Either motif sufficient (backup)',
        'No interaction detected'
    ]
})
print(grammar_table.to_string(index=False))
grammar_table.to_csv(f"{CONFIG['results_dir']}/tables/grammar_summary.csv", index=False)

# Table 5: Generative Validation
print("\n\nTable 5: Generative Validation Results\n")
if validation_results.get('success_rate', 0) > 0:
    gen_table = pd.DataFrame({
        'Metric': [
            'Generation Success Rate',
            'GC Content (Generated)',
            'GC Content (Real)',
            'KS Test p-value (GC)',
            'Motif Count (Generated)',
            'Motif Count (Real)',
            'Mann-Whitney p-value',
            'Overall Validation'
        ],
        'Value': [
            f"{validation_results['success_rate']:.1f}%",
            f"{np.mean(validation_results.get('generated_gc', [0])):.4f} ± {np.std(validation_results.get('generated_gc', [0])):.4f}",
            f"{np.mean(validation_results.get('real_gc', [0])):.4f} ± {np.std(validation_results.get('real_gc', [0])):.4f}",
            f"{validation_results.get('statistical_tests', {}).get('gc_kstest_pvalue', 0):.4f}",
            f"{np.mean(validation_results.get('generated_motifs', [0])):.2f} ± {np.std(validation_results.get('generated_motifs', [0])):.2f}",
            f"{np.mean(validation_results.get('real_motifs', [0])):.2f} ± {np.std(validation_results.get('real_motifs', [0])):.2f}",
            f"{validation_results.get('statistical_tests', {}).get('motif_mwtest_pvalue', 0):.4f}",
            'PASSED' if validation_results.get('validation_passed', False) else 'FAILED'
        ]
    })
else:
    gen_table = pd.DataFrame({
        'Metric': ['Generation Success Rate'],
        'Value': ['0.0%']
    })
print(gen_table.to_string(index=False))
gen_table.to_csv(f"{CONFIG['results_dir']}/tables/generative_validation.csv", index=False)

# Table 6: Model Configuration Summary
print("\n\nTable 6: Model Configuration Summary\n")
config_table = pd.DataFrame({
    'Parameter': [
        'GPU Type',
        'Transformer Layers',
        'Attention Heads',
        'Model Dimension',
        'Total Parameters',
        'Batch Size',
        'Training Epochs',
        'Samples per Epoch',
        'Mixed Precision',
        'Learning Rate'
    ],
    'Value': [
        GPU_TYPE,
        CONFIG['n_transformer_layers'],
        CONFIG['n_attention_heads'],
        CONFIG['d_model'],
        f"{model.get_num_params()/1e6:.1f}M",
        CONFIG['batch_size'],
        len(history['train_loss']),
        f"{CONFIG['samples_per_epoch']:,}",
        CONFIG['use_mixed_precision'],
        CONFIG['learning_rate']
    ]
})
print(config_table.to_string(index=False))
config_table.to_csv(f"{CONFIG['results_dir']}/tables/model_config.csv", index=False)

print("\n" + "="*70)
print("ALL TABLES SAVED")
print("="*70)
print(f"Location: {CONFIG['results_dir']}/tables/")
print("\n✓ 6 publication-ready tables generated")
print("✓ All tables saved in CSV format")

## 10. Comprehensive Research Report

Generate a publication-ready research summary report.

In [ ]:
# Generate Comprehensive Research Report
print("Generating comprehensive research report...")

report_lines = []
report_lines.append("# Enhancer Repair Research: Results Summary")
report_lines.append("=" * 70)
report_lines.append("")

# Section 1: Dataset Statistics
report_lines.append("## 1. Dataset Statistics")
report_lines.append("-" * 70)
report_lines.append(f"Total Sequences: {len(y):,}")
report_lines.append(f"Training Set: {len(train_idx):,} ({len(train_idx)/len(y)*100:.1f}%)")
report_lines.append(f"Validation Set: {len(val_idx):,} ({len(val_idx)/len(y)*100:.1f}%)")
report_lines.append(f"Test Set: {len(test_idx):,} ({len(test_idx)/len(y)*100:.1f}%)")
report_lines.append(f"Enhancers: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")
report_lines.append(f"Background: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
report_lines.append("")

# Section 2: Model Performance
report_lines.append("## 2. Model Performance")
report_lines.append("-" * 70)
report_lines.append(f"Architecture: Hybrid CNN-Transformer")
report_lines.append(f"Total Parameters: {model.get_num_params()/1e6:.1f}M")
report_lines.append(f"GPU Configuration: {GPU_TYPE}")
report_lines.append("")
report_lines.append("### Training Results")
report_lines.append(f"Final Train Loss: {history['train_loss'][-1]:.4f}")
report_lines.append(f"Final Train Accuracy: {history['train_acc'][-1]:.2f}%")
report_lines.append(f"Final Val Loss: {history['val_loss'][-1]:.4f}")
report_lines.append(f"Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
report_lines.append(f"Training Epochs: {len(history['train_loss'])}")
report_lines.append("")
report_lines.append("### Test Set Performance")
report_lines.append(f"Test Accuracy: {test_results['test_accuracy']:.2f}% "
                      f"(95% CI: [{test_results['accuracy_ci'][0]:.2f}%, {test_results['accuracy_ci'][1]:.2f}%])")
report_lines.append(f"ROC AUC: {test_results['auc']:.3f}")
report_lines.append(f"Precision: {test_results['precision']:.3f}")
report_lines.append(f"Recall: {test_results['recall']:.3f}")
report_lines.append(f"F1 Score: {test_results['f1']:.3f} "
                      f"(95% CI: [{test_results['f1_ci'][0]:.3f}, {test_results['f1_ci'][1]:.3f}])")
report_lines.append(f"Specificity: {test_results['specificity']:.3f}")
report_lines.append("")

# Section 3: Cross-Species Conservation
report_lines.append("## 3. Cross-Species Conservation Results")
report_lines.append("-" * 70)
for species_name, result in cross_species_results.items():
    if species_name != 'summary':
        report_lines.append(f"{species_name}: {result['repair_rate']:.1f}% "
                              f"({result['successes']}/{result['n_tested']} repaired)")
if 'summary' in cross_species_results:
    report_lines.append("")
    report_lines.append(f"Mammalian avg: {cross_species_results['summary'].get('mammal_rate', 0):.1f}%")
    report_lines.append(f"Non-mammalian avg: {cross_species_results['summary'].get('non_mammal_rate', 0):.1f}%")
    report_lines.append(f"Conservation score: {cross_species_results['summary'].get('conservation_score', 0):.1f}%")
report_lines.append("")

# Section 4: Grammar Discovery
report_lines.append("## 4. Grammar Discovery Results")
report_lines.append("-" * 70)
report_lines.append(f"Motif Regions Identified: {len(motif_regions)}")
report_lines.append(f"Motif Pair Interactions Tested: {len(interactions)}")
report_lines.append("")
report_lines.append("### Interaction Classification")
if len(interactions) > 0:
    report_lines.append(f"Synergistic: {len(synergistic)} ({len(synergistic)/len(interactions)*100:.1f}%)")
    report_lines.append(f"Redundant: {len(redundant)} ({len(redundant)/len(interactions)*100:.1f}%)")
    report_lines.append(f"Independent: {len(independent)} ({len(independent)/len(interactions)*100:.1f}%)")
else:
    report_lines.append("No interactions tested")
report_lines.append("")

# Section 5: Generative Validation
report_lines.append("## 5. Generative Validation Results")
report_lines.append("-" * 70)
if validation_results.get('success_rate', 0) > 0:
    report_lines.append(f"Generation Attempts: {validation_results['n_attempts']}")
    report_lines.append(f"Successful Generations: {validation_results['n_generated']}")
    report_lines.append(f"Success Rate: {validation_results['success_rate']:.1f}%")
    report_lines.append("")
    report_lines.append("### Statistical Validation")
    stats = validation_results.get('statistical_tests', {})
    report_lines.append(f"GC Content KS Test p-value: {stats.get('gc_kstest_pvalue', 0):.4f}")
    report_lines.append(f"Motif Count Mann-Whitney p-value: {stats.get('motif_mwtest_pvalue', 0):.4f}")
    report_lines.append(f"Overall Validation: {'PASSED' if validation_results.get('validation_passed', False) else 'FAILED'}")
else:
    report_lines.append("No sequences generated")
report_lines.append("")

# Section 6: Key Findings
report_lines.append("## 6. Key Findings for Publication")
report_lines.append("-" * 70)
report_lines.append("1. Model achieves high accuracy in distinguishing enhancers from background")
report_lines.append(f"   (Test AUC: {test_results['auc']:.3f})")
report_lines.append("")
report_lines.append("2. Cross-species repair analysis reveals evolutionary conservation")
if len(cross_species_results) > 0:
    avg_repair = np.mean([r['repair_rate'] for r in cross_species_results.values() if isinstance(r, dict) and 'repair_rate' in r])
    species_count = sum(1 for r in cross_species_results.values() if isinstance(r, dict) and 'repair_rate' in r)
    report_lines.append(f"   (Average repair rate: {avg_repair:.1f}% across {species_count} species)")
report_lines.append("")
report_lines.append("3. Grammar discovery identifies motif interaction patterns")
if len(interactions) > 0:
    report_lines.append(f"   ({len(synergistic)} synergistic, {len(redundant)} redundant interactions found)")
report_lines.append("")
report_lines.append("4. Generative validation demonstrates causal understanding")
if validation_results.get('success_rate', 0) > 0:
    report_lines.append(f"   (Success rate: {validation_results['success_rate']:.1f}%, "
                          f"validation: {'passed' if validation_results.get('validation_passed') else 'failed'})")
report_lines.append("")

# Section 7: Reproducibility
report_lines.append("## 7. Reproducibility Information")
report_lines.append("-" * 70)
report_lines.append(f"Random Seed: 42")
report_lines.append(f"PyTorch Version: {torch.__version__}")
report_lines.append(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    report_lines.append(f"GPU: {torch.cuda.get_device_name(0)}")
report_lines.append("")
report_lines.append("Model Configuration:")
report_lines.append(f"  - Transformer layers: {CONFIG['n_transformer_layers']}")
report_lines.append(f"  - Attention heads: {CONFIG['n_attention_heads']}")
report_lines.append(f"  - Model dimension: {CONFIG['d_model']}")
report_lines.append(f"  - CNN channels: {CONFIG['cnn_channels']}")
report_lines.append(f"  - Batch size: {CONFIG['batch_size']}")
report_lines.append(f"  - Learning rate: {CONFIG['learning_rate']}")
report_lines.append("")
report_lines.append("=" * 70)

# Print to console
report_text = "\n".join(report_lines)
print("\n" + report_text)

# Save to file
report_file = f"{CONFIG['results_dir']}/RESEARCH_SUMMARY_REPORT.md"
with open(report_file, 'w') as f:
    f.write(report_text)

print(f"\n✓ Comprehensive research report saved to: {report_file}")
print(f"✓ Report contains {len(report_lines)} lines covering all key findings")


## 12. Save Final Results


In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'history': history,
    'test_results': test_results
}, f"{CONFIG['project_dir']}/models/final_model.pt")

# Save all results
all_results = {
    'training_history': history,
    'test_results': test_results,
    'cross_species_results': cross_species_results,
    'grammar_results': {
        'motif_regions': motif_regions,
        'interactions': interactions,
        'synergistic': len(synergistic),
        'redundant': len(redundant),
        'independent': len(independent)
    },
    'generative_results': validation_results
}

with open(f"{CONFIG['project_dir']}/results/all_results.pkl", 'wb') as f:
    pickle.dump(all_results, f)

print(f"\nAll results saved to {CONFIG['project_dir']}")
print("\n" + "="*60)
print("RESEARCH COMPLETE!")
print("="*60)

---

## Notes for Resuming After Colab Timeout

If Colab disconnects, your checkpoints are saved to Google Drive.

To resume:
1. Re-run cells 1-4 (Environment setup)
2. Skip data generation (cell 3.2-3.4) if data already exists
3. Re-create model (cell 4.1)
4. Run training - it will auto-resume from checkpoint

```python
# Quick resume check
import os
checkpoint_path = f"{PROJECT_DIR}/checkpoints/checkpoint_latest.pt"
if os.path.exists(checkpoint_path):
    print(f"Checkpoint found! Training will resume.")
else:
    print("No checkpoint - starting fresh.")
```